### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="sberbank_housing_market_forecasting",
    dataset_year="2017",
    domain_str="business & marketing",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/competitions/sberbank-russian-housing-market",
    download_description="""
We use the data from Kaggle:

kaggle competitions download -c sberbank-russian-housing-market
mkdir -p local-data-warehouse/sberbank_housing_market_forecasting && mv sberbank-russian-housing-market.zip local-data-warehouse/sberbank_housing_market_forecasting/ && cd local-data-warehouse/sberbank_housing_market_forecasting/ && unzip sberbank-russian-housing-market.zip && rm test.csv.zip sample_submission.csv.zip data_dictionary.txt sberbank-russian-housing-market.zip
""",
    # References
    academic_reference_bibtex=r"""@misc{Herman2024HomeCreditCreditRiskModelStability,
  author = {Daniel Herman and Tomas Jelinek and Walter Reade and Maggie Demkin and Addison Howard},
  title  = {Home Credit - Credit Risk Model Stability},
  year   = {2024},
  howpublished = {\url{https://kaggle.com/competitions/home-credit-credit-risk-model-stability}},
  note   = {Kaggle competition}
}
""",
    academic_reference_bibtex_key="Herman2024HomeCreditCreditRiskModelStability",
    license="Kaggle Competition Rules",
    data_tags=["Non-IID", "Temporal", "Spatial"],
    curation_comments="""
We follow the preprocessing from TabRed (https://github.com/yandex-research/tabred/tree/main/preprocessing#sberbank-housing-market-forecasting).

- We drop two duplicated columns with the same values as other columns: "0_6_all", "7_14_all"
- We drop the ID column as it does not provide information.
- Note, the dataset contains a lot of spatial feature and even more already decoded spatial information (like distances to various points of interest).
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="price_doc",
    problem_type="regression",
    # We use RMSE instead of RMSLE from the competition as we already log scale the target
    objective_metric_name="rmse",
    time_on="timestamp",
)

## Preprocessing

In [2]:
import pandas as pd
import polars as pl
import zipfile
from pathlib import Path

# From: https://github.com/yandex-research/tabred/blob/main/preprocessing/sberbank-housing.py
data = pl.read_csv(
    zipfile.ZipFile(dataset_mold.path / 'train.csv.zip').read('train.csv'),
    null_values=["NA"],
    infer_schema_length=30_000
).with_columns(
    pl.col('timestamp').str.strptime(pl.Date)
)
data_macro = pl.read_csv(
    zipfile.ZipFile(dataset_mold.path / 'macro.csv.zip').read('macro.csv'),
    null_values=["NA"],
    infer_schema_length=30_000
).with_columns(
    pl.col('timestamp').str.strptime(pl.Date),
    pl.col('child_on_acc_pre_school').str.replace(',', '.').cast(pl.Float32, strict=False),
    pl.col('modern_education_share').str.replace(',', '.').cast(pl.Float32),
    pl.col('old_education_build_share').str.replace(',', '.').cast(pl.Float32),
).drop('provision_retail_space_modern_sqm') # this feature has one value except for nulls
data_fixup = (
    pl.read_excel(Path.cwd() / "BAD_ADDRESS_FIX.xlsx")
      .with_columns(pl.col(pl.Utf8).replace("NA", None))
)

data = data.filter(
    pl.col('kremlin_km').ne(pl.col('kremlin_km').min()) |
    pl.col('id').is_in(data_fixup['id'])
).update(data_fixup, on='id')
data = data.filter(
    pl.col('full_sq').gt(5.0) &
    pl.col('full_sq').ne(5326.0) &
    pl.col('price_doc').gt(1_000_000) &
    pl.col('price_doc').ne(2_000_000) &
    pl.col('price_doc').ne(3_000_000)
)

data = data.join(data_macro, on="timestamp")

# log scale target as in TabRed
data = data.with_columns(
    (pl.col("price_doc") / pl.col("full_sq")).log().alias("price_doc")
)

data = data.to_pandas()
cat_cols = [
    'ID_railroad_station_walk','ID_railroad_station_avto','ID_big_road1','ID_big_road2','ID_railroad_terminal','ID_bus_terminal','ecology','material','state','sub_area','product_type', 'culture_objects_top_25', 'thermal_power_plant_raion', 'incineration_raion', 'oil_chemistry_raion', 'radiation_raion', 'railroad_terminal_raion', 'big_market_raion', 'nuclear_reactor_raion', 'detention_facility_raion', 'water_1line', 'big_road1_1line', 'railroad_1line'
]
data[cat_cols] = data[cat_cols].astype("category")
data = data.drop(columns=["0_6_all", "7_14_all", "id"])
data = data.reset_index(drop=True)
df = data

/tmp/ipykernel_152759/946747977.py:29: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  data = data.filter(


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

TypeError: DataFrame contains object dtype columns: ['product_type', 'culture_objects_top_25', 'thermal_power_plant_raion', 'incineration_raion', 'oil_chemistry_raion', 'radiation_raion', 'railroad_terminal_raion', 'big_market_raion', 'nuclear_reactor_raion', 'detention_facility_raion', 'water_1line', 'big_road1_1line', 'railroad_1line']

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

date_col = task_mold.time_on
target_col = task_mold.target_column_name

df = df.sort_values(by=date_col).reset_index(drop=True)

n_steps = 5
now = df[date_col].max().normalize()

splits = {}

for step in range(n_steps):
    # Move split point back by 6 months each step
    test_time_max = now - pd.DateOffset(months=6 * step) + pd.DateOffset(day=1)
    test_time_min = now - pd.DateOffset(months=6 * (step + 1))

    if step == 0:
        test_mask = df[date_col] >= test_time_min
    else:
        test_mask = (df[date_col] >= test_time_min) & (df[date_col] < test_time_max)
    # Define indices
    test_idx = df.index[test_mask].to_numpy().tolist()
    train_idx = df.index[
        df[date_col] < test_time_min
    ].to_numpy().tolist()

    # Diagnostics
    print(f"\n=== Step {step} ===")
    print("Train size:", len(train_idx), "| Test size:", len(test_idx))
    print("Train target mean:", df.loc[train_idx, target_col].mean())
    print("Test target mean:", df.loc[test_idx, target_col].mean())

    splits[step] = {0: (train_idx, test_idx)}


splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="We always use 6 month of the data as test data and all prior data as training data. This simulate a model that is refit every half year. We create 5 splits going back 6 months each, starting from the newest date.",
    splits=splits,
    time_horizon=6,
    time_horizon_unit="months",
)

## Export

In [ ]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)